In [1]:
import pandas as pd
proofnet = pd.read_csv('/Users/aditya/neural-proof-assistant/backend/data/proofnet_train.csv')
proofnet.head()

,id,nl_statement,nl_proof,formal_statement,src_header
0,Rudin|exercise_1_1a,If $r$ is rational $(r \neq 0)$ and $x$ is irr...,\begin{proof}\n\n If $r$ and $r+x$ were bot...,theorem exercise_1_1a\n (x : ℝ) (y : ℚ) :\n ...,import .common\n\nopen real complex\nopen topo...
1,Rudin|exercise_1_2,Prove that there is no rational number whose s...,"\begin{proof}\n\n Suppose $m^2=12 n^2$, whe...","theorem exercise_1_2 : ¬ ∃ (x : ℚ), ( x ^ 2 = ...",import .common\n\nopen real complex\nopen topo...
2,Rudin|exercise_1_5,Let $A$ be a nonempty set of real numbers whic...,\begin{proof}\n\n We need to prove that $-\...,theorem exercise_1_5 (A minus_A : set ℝ) (hA :...,import .common\n\nopen real complex\nopen topo...
3,Rudin|exercise_1_11a,"If $z$ is a complex number, prove that there e...","\begin{proof}\n\n If $z=0$, we take $r=0, w...",theorem exercise_1_11a (z : ℂ) : \n ∃ (r : ℝ)...,import .common\n\nopen real complex\nopen topo...
4,Rudin|exercise_1_13,"If $x, y$ are complex, prove that $||x|-|y|| \...","\begin{proof}\n\n Since $x=x-y+y$, the tria...",theorem exercise_1_13 (x y : ℂ) : \n |(abs x)...,import .common\n\nopen real complex\nopen topo...


In [2]:
useful_cols = ['nl_statement', 'nl_proof']

In [3]:
proofnet_data = proofnet[useful_cols]

In [4]:
len(proofnet_data)

185

In [5]:
proofnet_data.isnull().sum()

nl_statement    0
nl_proof        1
dtype: int64

In [6]:
proofnet_data = proofnet_data.dropna()

In [7]:
proofnet_data.isnull().sum()

nl_statement    0
nl_proof        0
dtype: int64

In [8]:
proofnet_data

,nl_statement,nl_proof
0,If $r$ is rational $(r \neq 0)$ and $x$ is irr...,\begin{proof}\n\n If $r$ and $r+x$ were bot...
1,Prove that there is no rational number whose s...,"\begin{proof}\n\n Suppose $m^2=12 n^2$, whe..."
2,Let $A$ be a nonempty set of real numbers whic...,\begin{proof}\n\n We need to prove that $-\...
3,"If $z$ is a complex number, prove that there e...","\begin{proof}\n\n If $z=0$, we take $r=0, w..."
4,"If $x, y$ are complex, prove that $||x|-|y|| \...","\begin{proof}\n\n Since $x=x-y+y$, the tria..."
...,...,...
180,"Prove that $(x, y)$ is not a principal ideal i...","\begin{proof}\n\n Suppose, to the contrary,..."
181,Prove that if $f(x)$ and $g(x)$ are polynomial...,"\begin{proof}\n\n Let $f(x), g(x) \in \math..."
182,Prove that $x^6+30x^5-15x^3 + 6x-120$ is irred...,\begin{proof}\n\n $$\n\nx^6+30 x^5-15 x^3+6...
183,"Prove that $\frac{(x+2)^p-2^p}{x}$, where $p$ ...",\begin{proof}\n\n$\frac{(x+2)^p-2^p}{x} \quad ...


In [9]:
len(proofnet_data)

184

In [10]:
def extract_first_step(proof):
    '''
    extracts the first meaningful sentence from the proof.
    '''
    lines = proof.splitlines()
    for line in lines:
        line = line.replace("\\begin{proof}", "").replace("\\end{proof}", "").strip()
        if line:
            return line
    return ""


proofnet_data['first_step'] = proofnet_data['nl_proof'].apply(extract_first_step)

In [11]:
proofnet_data

,nl_statement,nl_proof,first_step
0,If $r$ is rational $(r \neq 0)$ and $x$ is irr...,\begin{proof}\n\n If $r$ and $r+x$ were bot...,"If $r$ and $r+x$ were both rational, then $x=r..."
1,Prove that there is no rational number whose s...,"\begin{proof}\n\n Suppose $m^2=12 n^2$, whe...","Suppose $m^2=12 n^2$, where $m$ and $n$ have n..."
2,Let $A$ be a nonempty set of real numbers whic...,\begin{proof}\n\n We need to prove that $-\...,We need to prove that $-\sup (-A)$ is the grea...
3,"If $z$ is a complex number, prove that there e...","\begin{proof}\n\n If $z=0$, we take $r=0, w...","If $z=0$, we take $r=0, w=1$. (In this case $w..."
4,"If $x, y$ are complex, prove that $||x|-|y|| \...","\begin{proof}\n\n Since $x=x-y+y$, the tria...","Since $x=x-y+y$, the triangle inequality gives"
...,...,...,...
180,"Prove that $(x, y)$ is not a principal ideal i...","\begin{proof}\n\n Suppose, to the contrary,...","Suppose, to the contrary, that $(x, y)=p$ for ..."
181,Prove that if $f(x)$ and $g(x)$ are polynomial...,"\begin{proof}\n\n Let $f(x), g(x) \in \math...","Let $f(x), g(x) \in \mathbb{Q}[x]$ be such tha..."
182,Prove that $x^6+30x^5-15x^3 + 6x-120$ is irred...,\begin{proof}\n\n $$\n\nx^6+30 x^5-15 x^3+6...,$$
183,"Prove that $\frac{(x+2)^p-2^p}{x}$, where $p$ ...",\begin{proof}\n\n$\frac{(x+2)^p-2^p}{x} \quad ...,$\frac{(x+2)^p-2^p}{x} \quad \quad p$ is on ad...


In [12]:
# set tactic
import re

TACTIC_PATTERNS = {
    "ASSUME": [r"\bsuppose\b", r"\bnassume\b", r"\bif\b.*then", r"\blet us suppose\b"],
    "CONTRADICTION": [r"\bcontradiction\b", r"\bassume.*not\b", r"\bsuppose.*not\b"],
    "LET": [r"^let\b", r"\bdefine\b"],
    "APPLY": [r"\bapply\b", r"\buse\b", r"from .* it follows"],
    "INDUCTION": [r"\binduction\b", r"by induction", r"base case"],
    "COMPUTE": [r"\bcompute\b", r"by calculation", r"evaluate"],
    "SIMPLIFY": [r"simplify", r"clearly", r"obviously"],
    "CONCLUDE": [r"\btherefore\b", r"we conclude", r"thus"],
}

def get_tactic(first_step: str) -> str:
    first_step = first_step.strip().lower()
    for tactic, patterns in TACTIC_PATTERNS.items():
        for pattern in patterns:
            if re.search(pattern, first_step):
                return tactic
    return "UNKNOWN"

proofnet_data['tactic'] = proofnet_data['first_step'].apply(get_tactic)

In [13]:
proofnet_data

,nl_statement,nl_proof,first_step,tactic
0,If $r$ is rational $(r \neq 0)$ and $x$ is irr...,\begin{proof}\n\n If $r$ and $r+x$ were bot...,"If $r$ and $r+x$ were both rational, then $x=r...",ASSUME
1,Prove that there is no rational number whose s...,"\begin{proof}\n\n Suppose $m^2=12 n^2$, whe...","Suppose $m^2=12 n^2$, where $m$ and $n$ have n...",ASSUME
2,Let $A$ be a nonempty set of real numbers whic...,\begin{proof}\n\n We need to prove that $-\...,We need to prove that $-\sup (-A)$ is the grea...,UNKNOWN
3,"If $z$ is a complex number, prove that there e...","\begin{proof}\n\n If $z=0$, we take $r=0, w...","If $z=0$, we take $r=0, w=1$. (In this case $w...",UNKNOWN
4,"If $x, y$ are complex, prove that $||x|-|y|| \...","\begin{proof}\n\n Since $x=x-y+y$, the tria...","Since $x=x-y+y$, the triangle inequality gives",UNKNOWN
...,...,...,...,...
180,"Prove that $(x, y)$ is not a principal ideal i...","\begin{proof}\n\n Suppose, to the contrary,...","Suppose, to the contrary, that $(x, y)=p$ for ...",ASSUME
181,Prove that if $f(x)$ and $g(x)$ are polynomial...,"\begin{proof}\n\n Let $f(x), g(x) \in \math...","Let $f(x), g(x) \in \mathbb{Q}[x]$ be such tha...",LET
182,Prove that $x^6+30x^5-15x^3 + 6x-120$ is irred...,\begin{proof}\n\n $$\n\nx^6+30 x^5-15 x^3+6...,$$,UNKNOWN
183,"Prove that $\frac{(x+2)^p-2^p}{x}$, where $p$ ...",\begin{proof}\n\n$\frac{(x+2)^p-2^p}{x} \quad ...,$\frac{(x+2)^p-2^p}{x} \quad \quad p$ is on ad...,UNKNOWN


In [14]:
proofnet_data.to_csv("proofnet.csv", index=False)